# BBM ↔ external-platform harmonization — findings

Extends *Breakdowns in Realizing the Digital Extended Specimen*, from the paper's 131-record sample to the full BBM fungal collection, and adds automated cross-platform record resolution.

**Harmonization relationships** (README): for a BBM specimen and its counterpart on a public platform —
- **bidirectional** — we cite their id *and* they cite our catalog number back
- **unidirectional** — only one side cites the other (either direction)
- **absent** — same specimen, cited nowhere in either direction

Every number below is produced live by calling the pipeline scripts. Network-dependent cells are marked; run the fetch scripts first (`get_bbm_records.py`, `get_mo_records.py`).

In [13]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "scripts" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import link_audit as la
print("repo:", ROOT)

repo: /Users/wfrankel/Desktop/breakdowns_DES


## Provenance — what this run actually fetched

Read from `config` + the CSVs on disk, so the numbers below reflect **this** run,
not a remembered one. The `.env` locality/collector filters apply to **BBM only**
(`get_bbm_records.py`); the external pulls are scoped by collection / dataset /
MO seed instead. If BBM filters are ON, the "34,856 / collection-wide" figures in
§1 no longer hold — the check at the bottom of the cell flags that.

In [14]:
import csv, datetime, config as cfg

def _info(name):
    p = cfg.DATA_DIR / name
    if not p.exists():
        return "— not fetched yet"
    with open(p, encoding="utf-8") as f:
        n = sum(1 for _ in f) - 1
    ts = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    return f"{n:>7,} rows   (fetched {ts})"

fc, fl = cfg.FILTER_COLLECTORS or [], cfg.FILTER_LOCALITY or []
bbm_filtered = bool(fc or fl)

print("BBM filters :", "OFF — full fungal collection"
      if not bbm_filtered else f"ON — collectors={fc} locality={fl}")
print("MO seeds    :", f"user={cfg.MO_USER or '—'}  location={cfg.MO_LOCATION or '—'}")
print("data dir    :", cfg.DATA_DIR)
print()
for n in ["bbm_records.csv", "mo_records.csv", "mycoportal_records.csv",
          "gbif_records.csv", "genbank_records.csv"]:
    print(f"  {n:<24} {_info(n)}")

if bbm_filtered:
    print("\n\u26a0  BBM filters are ON \u2014 the \'34,856 / collection-wide\' "
          "figures in \u00a71 assume NO filters and no longer apply.")

BBM filters : OFF — full fungal collection
MO seeds    : user=2873  location=1679
data dir    : /Users/wfrankel/Desktop/breakdowns_DES/data

  bbm_records.csv           34,940 rows   (fetched 2026-09-03 10:56)
  mo_records.csv             5,866 rows   (fetched 2026-09-04 11:23)
  mycoportal_records.csv    34,946 rows   (fetched 2026-09-04 11:32)
  gbif_records.csv          34,878 rows   (fetched 2026-09-04 12:22)
  genbank_records.csv          500 rows   (fetched 2026-09-04 11:28)


## 1. BBM → Mushroom Observer (lookup by stored id)

MO is an **independent, upstream** platform — the Ceskas posted there directly, so MO holds no copy of our GUID. The only link is the id we recorded (`MO # 82752`). We scan `bbm_records.csv` for those, look each up on MO, and check whether MO cites us back (a free-text `UBC F#` note).

In [15]:
# NETWORK: queries Mushroom Observer
mo = la.MushroomObserver()
res = la.audit(mo)                     # scan -> lookup -> classify
c = res["counts"]
on_mo = c["bidirectional"] + c["unidirectional"]
print(f"BBM records scanned     : {res['n_rows']}")
print(f"records citing an MO id  : {res['n_with_ref']}  ({100*res['n_with_ref']/res['n_rows']:.2f}%)")
print(f"distinct MO ids          : {len(res['ref_map'])}")
print(f"  resolve on MO          : {on_mo}")
print(f"    bidirectional        : {c['bidirectional']}")
print(f"    unidirectional UBC→MO: {c['unidirectional']}")
print(f"  dangling               : {c['dangling']}")

BBM records scanned     : 34856
records citing an MO id  : 21  (0.06%)
distinct MO ids          : 20
  resolve on MO          : 20
    bidirectional        : 17
    unidirectional UBC→MO: 3
  dangling               : 0


## 2. MyCoPortal (harvested — matched by GUID)

MyCoPortal is the **opposite coupling**: it is **harvested wholesale from our Specify database** via Symbiota, so every MP record carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. The `Mycoportal # UBC#####` strings in our records are **legacy free-text, not queryable ids** (they even ride along verbatim into MP's `occurrenceRemarks`). The reliable link is the **GUID**.

Confirmed against the live Symbiota API (`/api/v2/occurrence?occurrenceID=<guid>`):
- UBC fungi on MyCoPortal (`collid 49`): **34,946 records** ≈ our 34,856 — essentially the whole collection.
- match: MP `occurrenceID` == BBM `guid`; MP `catalogNumber` == BBM `F#`.

**Coupling contrast, quantified:** loosely-coupled MO → **0.06 %** cross-referenced; tightly-coupled MyCoPortal → **~complete**. That is the paper's central argument in two numbers. (A GUID-discovery audit that counts MP presence + harvest gaps per record is the natural next script.)

In [16]:
# OFFLINE: how many records carry the legacy 'Mycoportal #' annotation
mp = la.MyCoPortal()
ref_map, n_rows, n_with_ref = la.scan(mp, str(la.INPUT))
print(f"records with a legacy 'Mycoportal #' note : {n_with_ref} (of {n_rows})")
print("→ not a queryable id; real MP linkage is the GUID (see above)")

records with a legacy 'Mycoportal #' note : 75 (of 34856)
→ not a queryable id; real MP linkage is the GUID (see above)


## 3. Ceska / Observatory Hill quadrants — cross-platform resolution (README §2)

To answer *how many MO records are ours but unconnected* we resolve records **by attributes**, using the evaluator subsystem **vendored** into `scripts/evaluators/` (copied from the orchestration framework) — `RuleBasedEvaluator` + `LLMEvaluator` — via `resolve.py`. Both platforms are shaped into orchestration's record contract, blocked by genus, and clustered; a cluster with a BBM and an MO record is a match. Each match is then scored into a quadrant by crossing the attribute match with the recorded cross-references.

**Seeds:** MO user 2873 (Ceska) = 5,866 obs; MO location 1679 (Observatory Hill) = 2,707 obs.

> Requires `bbm_records.csv` (joined) and `mo_records.csv`. Set `LLM_MODEL` in `.env` to enable the LLM tier.

In [17]:
import resolve as R, platforms as P
from collections import Counter
mo = P.PLATFORMS["mo"]
bbm_p, mo_p = R.DATA_DIR / "bbm_records.csv", R.DATA_DIR / "mo_records.csv"

if bbm_p.exists() and mo_p.exists():
    bbm_rows, bmeta = R.load_bbm(str(bbm_p), mo)
    plat_rows, pmeta = R.load_platform(str(mo_p))
    meta = {**bmeta, **pmeta}
    pairs = R.resolve(bbm_rows, plat_rows, meta, use_llm=True)   # NETWORK if LLM_MODEL set
    q = Counter(R.quadrant(b, m, meta) for b, m, _ in pairs)
    how = Counter(h for _, _, h in pairs)
    print(f"BBM records            : {len(bbm_rows)}")
    print(f"MO Ceska/OH records    : {len(plat_rows)}")
    print(f"cross-platform matches : {len(pairs)}   (by method: {dict(how)})")
    for k in ("bidirectional","unidirectional_ubc_to_platform","unidirectional_platform_to_ubc","absent"):
        print(f"  {k:34} {q.get(k,0)}")
else:
    print("Run get_bbm_records.py and get_mo_records.py first, then re-run this cell.")

BBM records            : 34856
MO Ceska/OH records    : 5866
cross-platform matches : 21828   (by method: {'similar': 21828})
  bidirectional                      13
  unidirectional_ubc_to_platform     2
  unidirectional_platform_to_ubc     626
  absent                             21187


## 4. Figure 2 comparison

In [18]:
import pandas as pd
paper = {"UBC records citing MO":19, "bidirectional":8,
         "unidirectional (UBC→MO)":10, "dangling / wrong id":1, "bidirectional rate":"42%"}
ours  = {"UBC records citing MO":len(res["ref_map"]), "bidirectional":c["bidirectional"],
         "unidirectional (UBC→MO)":c["unidirectional"], "dangling / wrong id":c["dangling"],
         "bidirectional rate":f"{100*c['bidirectional']/max(len(res['ref_map']),1):.0f}%"}
pd.DataFrame({"Kholmatova 2026 (Fig 2, Phase I)":paper, "This audit (full collection)":ours})

,"Kholmatova 2026 (Fig 2, Phase I)",This audit (full collection)
UBC records citing MO,19,20
bidirectional,8,17
unidirectional (UBC→MO),10,3
dangling / wrong id,1,0
bidirectional rate,42%,85%


## 5. GBIF (harvested — matched by GUID)

GBIF is the **same coupling as MyCoPortal**: our Specify collection is **harvested wholesale** into GBIF as a published dataset (`datasetKey ca1bcd7e-7387-42f9-81ba-1470db55e3e8`), so every GBIF occurrence carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. Our records almost never store a GBIF id in free text — the reliable link is again the **GUID**, discovered from GBIF's side, not cited from ours.

- **Reference direction (below):** BBM free text → GBIF id. Expected ~0 — GBIF ids are not something a curator types into a specimen record.
- **Discovery direction (the real one):** `python scripts/get_records.py --platform gbif` pulls the whole dataset by `datasetKey` (~35k occurrences) and matches each back to a BBM record by `occurrenceID == guid`. Like MyCoPortal, presence should be **~complete**, so the finding is the *harvest gap* — our records with no GBIF twin — not missing cross-references.

In [19]:
# OFFLINE: GBIF ids appearing in BBM free text (real link is the GUID, discovered from GBIF)
gbif = la.PLATFORMS["gbif"]
ref_map, n_rows, n_with_ref = la.scan(gbif, str(la.INPUT))
print(f"BBM records scanned                 : {n_rows}")
print(f"records citing a GBIF id in free text: {n_with_ref}  (harvested platform → expected ~0)")
print(f"coupling                            : {gbif.coupling}  (matched by GUID / occurrenceID)")
print(f"GBIF dataset key                    : {gbif.DATASET_KEY}")
print("→ discovery direction: python scripts/get_records.py --platform gbif  (~35k occurrences)")

BBM records scanned                 : 34856
records citing a GBIF id in free text: 0  (harvested platform → expected ~0)
coupling                            : harvested  (matched by GUID / occurrenceID)
GBIF dataset key                    : ca1bcd7e-7387-42f9-81ba-1470db55e3e8
→ discovery direction: python scripts/get_records.py --platform gbif  (~35k occurrences)


## 6. GenBank (independent — matched by stored accession)

GenBank is coupled like **Mushroom Observer**: an **independent upstream** database where the link exists only if *our* record stores the sequence **accession** (`GenBank KX691234`), and it is *bidirectional* only if the GenBank record's own definition/notes cite our `UBC F#` back. Nothing is harvested, so — exactly as the paper predicts for loosely-coupled platforms — coverage is **sparse**: only sequenced specimens have an accession at all.

- **Reference direction (below):** BBM free text → GenBank accession, then look each up via NCBI eutils and check for a `UBC F#` back-reference. This is cheap: only records that actually cite an accession trigger a lookup.
- **Discovery direction:** `python scripts/get_records.py --platform genbank` searches NCBI for `UBC` / `University of British Columbia` vouchers — the way to find sequences that exist but were never linked from our side.

In [20]:
# NETWORK: only BBM records that cite a GenBank accession are looked up (expected sparse)
gb = la.PLATFORMS["genbank"]
res = la.audit(gb)
c = res["counts"]
print(f"BBM records scanned              : {res['n_rows']}")
print(f"records citing a GenBank accession: {res['n_with_ref']}")
print(f"distinct accessions               : {res['n_ids']}")
print(f"  bidirectional (cites UBC back)  : {c['bidirectional']}")
print(f"  unidirectional (UBC→GenBank)    : {c['unidirectional']}")
print(f"  dangling (accession not found)  : {c['dangling']}")
print("→ discovery direction: python scripts/get_records.py --platform genbank")

BBM records scanned              : 34856
records citing a GenBank accession: 336
distinct accessions               : 331
  bidirectional (cites UBC back)  : 2
  unidirectional (UBC→GenBank)    : 248
  dangling (accession not found)  : 81
→ discovery direction: python scripts/get_records.py --platform genbank


## Methods & caveats

- **BBM data**: full `collectionobject` table + joins to determination→taxon, collector→agent, collecting-event→locality (`get_bbm_records.py`).
- **MO reference formats caught**: `MO # 82752`, `MUOB 12345`, `Mushroom Observer observation #…`, mushroomobserver.org URLs. `MO posted as …` (no number) is a link with no id and is not looked up.
- **Coupling dictates method**: harvested-downstream platforms (MyCoPortal, GBIF) match by our GUID; independent platforms (MO, GenBank) match by the id we stored + attribute resolution.
- **Resolution**: rule-based predicates use name + exact date + locality/collector *token overlap* (cross-platform strings are formatted differently); the LLM tier adjudicates the ambiguous middle when configured.
- **Coverage**: all five platforms are wired through the shared `Platform` abstraction — MO (§1), MyCoPortal (§2), GBIF (§5), GenBank (§6), plus attribute resolution (§3). Remaining work is running the harvested-side **discovery audits** (`get_records.py --platform gbif|mycoportal`) to quantify harvest gaps, and the GenBank voucher search for unlinked sequences.